# Multi-Object Vehicle Tracking with Ultralytics YOLO

This notebook demonstrates **multi-object tracking** using a custom-trained YOLO model (`vehicle_detection_best.pt`) that detects the following vehicle classes:

| ID | Class |
|----|-------|
| 0  | Bicycle |
| 1  | Bus |
| 2  | Car |
| 3  | Heavy truck |
| 4  | Light truck |
| 5  | Motorcycle |
| 6  | Person |
| 7  | Semi-trailer / Combination vehicle |

Tracking extends object detection by maintaining a **unique ID** for each detected object across video frames. Ultralytics YOLO supports two built-in trackers:

- **BoT-SORT** (`botsort.yaml`) — default tracker, supports ReID
- **ByteTrack** (`bytetrack.yaml`) — fast, lightweight tracker

> 📖 Reference: [Ultralytics Track Docs](https://docs.ultralytics.com/modes/track/)

## 1. Install Dependencies

In [1]:
%pip install ultralytics opencv-python-headless --quiet

Note: you may need to restart the kernel to use updated packages.


## 2. Setup — Load Model and Configure Video Source

In [2]:
from ultralytics import YOLO

# Load the custom-trained vehicle detection model
model = YOLO("vehicle_detection_best.pt")

# ── Configure your video source ──────────────────────────────────────────────
# Replace this path with your actual video file, RTSP stream, or webcam index.
VIDEO_SOURCE = "/Users/sondre/kjorbekk/Kjorbekk_3000965_06.mp4"   # e.g. "traffic.mp4", 0 for webcam

print("Model loaded successfully.")
print(f"Classes: {model.names}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/sondre/opt/anaconda3/envs/svv_yolo/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/sondre/opt/anaconda3/envs/svv_yolo/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/sondre/opt/anaconda3/envs/svv_yolo/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 7

Model loaded successfully.
Classes: {0: 'bicycle', 1: 'bus', 2: 'car', 3: 'heavy truck', 4: 'light truck', 5: 'motorcycle', 6: 'person', 7: 'semi-trailer / combination vehicle'}


## 3. Basic Tracking — Default Tracker (BoT-SORT)

The simplest way to run tracking. YOLO assigns a persistent **track ID** to each detected vehicle.  
BoT-SORT is the default tracker; it supports optional Re-Identification (ReID) for improved accuracy across occlusions.

In [3]:
from ultralytics import YOLO

model = YOLO("vehicle_detection_best.pt")

# Run tracking with the default BoT-SORT tracker
# tracker="botsort.yaml" is the default — can be omitted
results = model.track(
    source=VIDEO_SOURCE,
    tracker="botsort.yaml",   # default tracker
    show=True,                # display video window
    save=True,                # save annotated output video
    conf=0.3,                 # minimum detection confidence
    iou=0.5,                  # IoU threshold for NMS
)

print("Tracking complete. Results saved to runs/track/")

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.5 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 8.5 MB/s  0:00:00

requirements: AutoUpdate success ✅ 2.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



RuntimeError: Numpy is not available

## 4. Tracking with ByteTrack

**ByteTrack** is a fast, lightweight tracker that uses both high- and low-confidence detections for improved association.  
Enable it by passing `tracker="bytetrack.yaml"`.

In [4]:
from ultralytics import YOLO

model = YOLO("vehicle_detection_best.pt")

# Run tracking with ByteTrack
results = model.track(
    source=VIDEO_SOURCE,
    tracker="bytetrack.yaml",  # ByteTrack tracker
    show=True,
    save=True,
    conf=0.3,
    iou=0.5,
)

print("ByteTrack tracking complete. Results saved to runs/track/")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



RuntimeError: Numpy is not available

## 5. Frame-by-Frame Tracking Loop (with OpenCV)

This approach processes the video frame-by-frame using OpenCV.  
The `persist=True` argument tells the tracker that each frame is part of a continuous sequence, so it can maintain track IDs across frames.

This is the recommended pattern when you need access to individual frames for custom processing (e.g., counting vehicles, saving crops, etc.).

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO("vehicle_detection_best.pt")

cap = cv2.VideoCapture(VIDEO_SOURCE)

while cap.isOpened():
    success, frame = cap.read()

    if not success:
        break  # End of video

    # Run YOLO tracking on the current frame; persist=True maintains track IDs
    results = model.track(frame, persist=True, conf=0.3, iou=0.5)

    # Draw bounding boxes and track IDs on the frame
    annotated_frame = results[0].plot()

    cv2.imshow("Vehicle Tracking — BoT-SORT", annotated_frame)

    # Press 'q' to quit early
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## 6. Plotting Vehicle Trajectories Over Time

Visualise the movement path of each tracked vehicle by drawing a trailing polyline from its centre point.  
`track_history` stores the last 60 centre-point positions per track ID.

In [ ]:
from collections import defaultdict

import cv2
import numpy as np
from ultralytics import YOLO

model = YOLO("vehicle_detection_best.pt")

cap = cv2.VideoCapture(VIDEO_SOURCE)

# Store centre-point history per track ID
track_history = defaultdict(list)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # Run tracking — persist=True keeps track IDs consistent across frames
    result = model.track(frame, persist=True, conf=0.3, iou=0.5)[0]

    if result.boxes is not None and result.boxes.id is not None:
        boxes    = result.boxes.xywh.cpu()           # (x_center, y_center, w, h)
        track_ids = result.boxes.id.int().cpu().tolist()
        class_ids = result.boxes.cls.int().cpu().tolist()

        # Draw detections on frame
        frame = result.plot()

        for box, track_id, cls_id in zip(boxes, track_ids, class_ids):
            x, y, w, h = box
            track = track_history[track_id]
            track.append((float(x), float(y)))

            # Keep only the last 60 frames of history
            if len(track) > 60:
                track.pop(0)

            # Draw trajectory line
            if len(track) > 1:
                points = np.array(track, dtype=np.int32).reshape((-1, 1, 2))
                cv2.polylines(frame, [points], isClosed=False, color=(0, 200, 255), thickness=2)

    cv2.imshow("Vehicle Trajectories", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## 7. Saving Tracking Output to a Video File

Annotated tracking results can be saved to disk automatically with `save=True`, or manually with OpenCV's `VideoWriter`.

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO("vehicle_detection_best.pt")

cap = cv2.VideoCapture(VIDEO_SOURCE)

# Read video properties for the writer
fps    = cap.get(cv2.CAP_PROP_FPS) or 25
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

OUTPUT_PATH = "tracked_output.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    results = model.track(frame, persist=True, conf=0.3, iou=0.5)
    annotated_frame = results[0].plot()

    writer.write(annotated_frame)

cap.release()
writer.release()
print(f"Saved annotated video to: {OUTPUT_PATH}")

## 8. Tracker Configuration Reference

### Available Trackers

| Tracker | Config file | Notes |
|---------|------------|-------|
| **BoT-SORT** | `botsort.yaml` | Default. Supports ReID for appearance-based re-identification. |
| **ByteTrack** | `bytetrack.yaml` | Fast. Uses both high- and low-confidence detections. |

### Key Tracker Arguments

| Parameter | Range | Description |
|-----------|-------|-------------|
| `track_high_thresh` | 0.0–1.0 | First-association confidence threshold. Detections below this are not used for primary matching. |
| `track_low_thresh` | 0.0–1.0 | Second-association threshold (more lenient fallback). |
| `new_track_thresh` | 0.0–1.0 | Minimum confidence to initialise a new track. |
| `track_buffer` | ≥ 0 | Frames to keep a lost track alive before deletion. Higher = more occlusion tolerance. |
| `match_thresh` | 0.0–1.0 | IoU threshold for matching detections to existing tracks. |
| `gmc_method` | orb / sift / ecc / sparseOptFlow / None | Global Motion Compensation method. Compensates for camera movement. |
| `with_reid` | True / False | Enable ReID (BoT-SORT only). Uses appearance embeddings to re-identify vehicles. |
| `proximity_thresh` | 0.0–1.0 | Minimum IoU required before ReID is applied. |
| `appearance_thresh` | 0.0–1.0 | Minimum visual similarity score for ReID matching. |

### Common `model.track()` Arguments

| Argument | Default | Description |
|----------|---------|-------------|
| `conf` | 0.25 | Detection confidence threshold. |
| `iou` | 0.7 | NMS IoU threshold. |
| `persist` | False | Set `True` when processing video frame-by-frame to maintain track IDs. |
| `tracker` | `botsort.yaml` | Tracker config file. |
| `save` | False | Save annotated output to `runs/track/`. |
| `show` | False | Display live video window. |
| `stream` | False | Return a generator (memory-efficient for long videos). |